# Early Sepsis Onset Prediction: LSTM vs Logistic Regression
### PhysioNet Computing in Cardiology Challenge 2019 — `dataset.csv`

---
**Objective:** Predict sepsis onset 6 hours in advance using PyTorch LSTM and Logistic Regression.

**Dataset:** Pre-merged PhysioNet 2019 CSV  
- Columns: 40 clinical features + `Hour` + `SepsisLabel` + `Patient_ID`  
- `Hour` resets to 0 for each patient (ICU admission hour)  
- `Patient_ID` uniquely identifies each patient  

**Key change from v1:** LSTM replaced with PyTorch implementation — real BPTT, batched forward pass, GPU support. Trains in minutes instead of hours.
---

## 0. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
    recall_score, precision_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print(f'NumPy {np.__version__} | Pandas {pd.__version__} | PyTorch {torch.__version__}')
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────
CSV_PATH = './dataset.csv'   # <-- path to your dataset.csv
# ─────────────────────────────────────────────────────────────

SEQUENCE_LEN       = 12     # lookback window in hours fed into LSTM
PREDICTION_HORIZON = 6      # predict sepsis this many hours ahead
TEST_SIZE          = 0.20
VAL_SIZE           = 0.10
BATCH_SIZE         = 256    # larger batches = faster GPU utilisation
EPOCHS             = 30
LEARNING_RATE      = 0.001
HIDDEN_SIZE        = 64
NUM_LAYERS         = 2      # stacked LSTM layers
DROPOUT            = 0.3

VITAL_FEATURES = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
LAB_FEATURES   = [
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST',
    'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
    'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium',
    'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI',
    'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets'
]
DEMO_FEATURES  = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS']
ALL_FEATURES   = VITAL_FEATURES + LAB_FEATURES + DEMO_FEATURES

print(f'Lookback: {SEQUENCE_LEN}h | Horizon: {PREDICTION_HORIZON}h | Base features: {len(ALL_FEATURES)}')

## 2. Load Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)

# Drop junk index column if present
df.drop(columns=[c for c in ['Unnamed: 0', 'Hour'] if c in df.columns], inplace=True)

# Validate required columns
required = ['Patient_ID', 'ICULOS', 'SepsisLabel']
missing_req = [c for c in required if c not in df.columns]
if missing_req:
    raise ValueError(f'Required columns missing from CSV: {missing_req}')

df.sort_values(['Patient_ID', 'ICULOS'], inplace=True)
df.reset_index(drop=True, inplace=True)

missing_feats = [f for f in ALL_FEATURES if f not in df.columns]
if missing_feats:
    print(f'WARNING: features absent from CSV and will be skipped: {missing_feats}')
    ALL_FEATURES = [f for f in ALL_FEATURES if f in df.columns]

print(f'Loaded  : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Patients: {df["Patient_ID"].nunique():,}')
print(f'Columns : {df.columns.tolist()}')

## 3. Exploratory Data Analysis

In [ ]:
patient_labels = df.groupby('Patient_ID')['SepsisLabel'].max()
n_patients  = patient_labels.shape[0]
n_sepsis    = int(patient_labels.sum())
prevalence  = n_sepsis / n_patients

print('=' * 52)
print('DATASET SUMMARY')
print('=' * 52)
print(f'Total patients        : {n_patients:,}')
print(f'Sepsis patients       : {n_sepsis:,}  ({prevalence:.1%})')
print(f'Non-sepsis patients   : {n_patients - n_sepsis:,}  ({1-prevalence:.1%})')
print(f'Total hourly rows     : {len(df):,}')
los = df.groupby('Patient_ID').size()
print(f'Avg ICU hours/patient : {los.mean():.1f}  (min {los.min()}, max {los.max()})')
print(f'Avg missingness (labs): {df[LAB_FEATURES].isna().mean().mean():.1%}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('PhysioNet 2019 — Exploratory Data Analysis', fontsize=14, fontweight='bold')

# 1. Class imbalance — patient level
ax = axes[0, 0]
counts = patient_labels.value_counts().sort_index()
bars = ax.bar(['Non-Sepsis', 'Sepsis'], counts.values,
               color=['#378ADD', '#D85A30'], alpha=0.85, width=0.5)
for bar, cnt in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + n_patients * 0.005,
            f'{int(cnt):,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution (Patient Level)')
ax.set_ylabel('Patients')

# 2. Top-15 features by missingness
ax = axes[0, 1]
miss = df[ALL_FEATURES].isna().mean().sort_values(ascending=False).head(15)
ax.barh(miss.index[::-1], miss.values[::-1] * 100, color='#534AB7', alpha=0.8)
ax.set_xlabel('Missing (%)')
ax.set_title('Top 15 Features by Missingness')

# 3. ICU LOS distribution
ax = axes[0, 2]
ax.hist(los, bins=50, color='#1D9E75', alpha=0.85, edgecolor='white', linewidth=0.4)
ax.axvline(los.mean(), color='#D85A30', linestyle='--', label=f'Mean {los.mean():.1f}h')
ax.set_xlabel('ICU Length of Stay (hours)')
ax.set_ylabel('Patients')
ax.set_title('ICU LOS Distribution')
ax.legend()

# 4. HR distribution
ax = axes[1, 0]
for lbl, col, name in [(0, '#378ADD', 'Non-Sepsis'), (1, '#D85A30', 'Sepsis')]:
    vals = df[df['SepsisLabel'] == lbl]['HR'].dropna()
    ax.hist(vals, bins=60, alpha=0.6, color=col, label=name, density=True)
ax.set_xlabel('Heart Rate (bpm)')
ax.set_title('HR: Sepsis vs Non-Sepsis')
ax.legend()

# 5. Lactate distribution
ax = axes[1, 1]
feat = 'Lactate' if 'Lactate' in df.columns else 'MAP'
for lbl, col, name in [(0, '#378ADD', 'Non-Sepsis'), (1, '#D85A30', 'Sepsis')]:
    vals = df[df['SepsisLabel'] == lbl][feat].dropna()
    ax.hist(vals, bins=60, alpha=0.6, color=col, label=name, density=True)
ax.set_xlabel(feat)
ax.set_title(f'{feat}: Sepsis vs Non-Sepsis')
ax.legend()

# 6. Sepsis label rate by ICU hour
ax = axes[1, 2]
hourly = df.groupby('ICULOS')['SepsisLabel'].mean().head(72)
ax.plot(hourly.index, hourly.values * 100, color='#D85A30', lw=2)
ax.fill_between(hourly.index, hourly.values * 100, alpha=0.2, color='#D85A30')
ax.set_xlabel('ICU Hour (ICULOS)')
ax.set_ylabel('Sepsis rate (%)')
ax.set_title('Sepsis Label Rate by ICU Hour')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved eda_plots.png')

## 4. Preprocessing & Feature Engineering

In [ ]:
def create_early_labels(df, horizon=6):
    """
    For each pre-onset hour t, EarlyLabel[t] = 1 if sepsis occurs
    within the next `horizon` hours. Rows already in sepsis
    (SepsisLabel == 1) are marked -1 and dropped.
    Fixed for Pandas 3.x — avoids groupby().apply() key-dropping bug.
    """
    df = df.sort_values(['Patient_ID', 'ICULOS']).copy()

    def _make_early(grp):
        s = grp['SepsisLabel'].values.astype(int)
        early = np.zeros(len(s), dtype=int)
        for t in range(len(s)):
            if s[t] == 1:
                early[t] = -1
            else:
                early[t] = int(s[t : t + horizon].sum() > 0)
        return early

    # Plain for-loop over groups — bypasses pandas 3.x apply() key-drop issue
    groups = df.groupby('Patient_ID', sort=False)
    early_labels = np.concatenate([_make_early(grp) for _, grp in groups])
    df['EarlyLabel'] = early_labels

    df = df[df['EarlyLabel'] != -1].copy().reset_index(drop=True)
    pos = df['EarlyLabel'].sum()
    print(f'Early labels ({horizon}h horizon): '
          f'{pos:,} positive / {len(df):,} rows  ({pos/len(df):.2%})')
    return df


def engineer_features(df):
    """Add composite clinical indicators derived from routine vitals."""
    df = df.copy()

    if {'HR', 'SBP'}.issubset(df.columns):
        df['ShockIndex'] = df['HR'] / df['SBP'].replace(0, np.nan)

    if {'Resp', 'SBP'}.issubset(df.columns):
        df['qSOFA_Resp']  = (df['Resp'] > 22).astype(float)
        df['qSOFA_SBP']   = (df['SBP']  < 100).astype(float)
        df['qSOFA_score'] = df['qSOFA_Resp'] + df['qSOFA_SBP']

    if 'Temp' in df.columns:
        df['Temp_abnormal'] = ((df['Temp'] > 38.3) | (df['Temp'] < 36.0)).astype(float)

    if 'MAP' in df.columns:
        df['Hypotension'] = (df['MAP'] < 65).astype(float)

    # Rolling features — plain loop avoids groupby.apply() issues
    for feat in ['HR', 'Resp', 'MAP', 'O2Sat']:
        if feat in df.columns:
            rolled = []
            for _, grp in df.groupby('Patient_ID', sort=False):
                rolled.append(grp[feat].rolling(3, min_periods=1).mean())
            df[f'{feat}_roll3'] = pd.concat(rolled).reindex(df.index)

    print(f'Feature engineering done. Shape: {df.shape}')
    return df


print('Step 1 — creating early prediction labels...')
processed = create_early_labels(df, horizon=PREDICTION_HORIZON)

print('Step 2 — engineering features...')
processed = engineer_features(processed)

ENGINEERED = ['ShockIndex', 'qSOFA_score', 'Temp_abnormal', 'Hypotension',
              'HR_roll3', 'Resp_roll3', 'MAP_roll3', 'O2Sat_roll3']

EXCLUDE = {'Patient_ID', 'ICULOS', 'SepsisLabel', 'EarlyLabel'}
FINAL_FEATURES = [
    f for f in (ALL_FEATURES + ENGINEERED)
    if f in processed.columns and f not in EXCLUDE
]

zero_var = [f for f in FINAL_FEATURES if processed[f].nunique(dropna=True) <= 1]
if zero_var:
    print(f'Dropping zero-variance features: {zero_var}')
    FINAL_FEATURES = [f for f in FINAL_FEATURES if f not in zero_var]

print(f'\nFinal feature count: {len(FINAL_FEATURES)}')
print(FINAL_FEATURES)

## 5. Sequence Construction

In [ ]:
def build_sequences(df, features, seq_len=12, label_col='EarlyLabel'):
    """
    Sliding-window sequence builder grouped by Patient_ID.
    Returns:
        X_seq  : (N, seq_len, F)  for LSTM
        X_flat : (N, F)           last timestep, for Logistic Regression
        y      : (N,)             binary labels
    """
    X_seq_list, X_flat_list, y_list = [], [], []
    skipped = 0

    for pid, grp in df.sort_values(['Patient_ID', 'ICULOS']).groupby('Patient_ID'):
        feat_arr = grp[features].values.astype(np.float32)
        labels   = grp[label_col].values
        T = len(grp)

        if T <= seq_len:
            skipped += 1
            continue

        for t in range(seq_len, T):
            X_seq_list.append(feat_arr[t - seq_len : t])
            X_flat_list.append(feat_arr[t - 1])
            y_list.append(int(labels[t]))

    if not X_seq_list:
        raise ValueError(
            f'No sequences built — all patients have <= {seq_len} usable hours.\n'
            f'Try lowering SEQUENCE_LEN (currently {seq_len}).'
        )

    X_seq  = np.array(X_seq_list,  dtype=np.float32)
    X_flat = np.array(X_flat_list, dtype=np.float32)
    y      = np.array(y_list,      dtype=np.int32)

    print(f'Sequences : {X_seq.shape}')
    print(f'Skipped patients (too short): {skipped}')
    print(f'Positive label rate: {y.mean():.2%}')

    if y.sum() == 0:
        raise ValueError('Zero positive labels — check SepsisLabel column and PREDICTION_HORIZON.')

    return X_seq, X_flat, y


print('Building sequences...')
X_seq, X_flat, y = build_sequences(processed, FINAL_FEATURES, seq_len=SEQUENCE_LEN)

In [ ]:
# ── Train / Val / Test split (stratified) ────────────────────
idx = np.arange(len(y))

idx_tv, idx_test = train_test_split(idx, test_size=TEST_SIZE,
                                    random_state=42, stratify=y)
idx_train, idx_val = train_test_split(idx_tv,
                                      test_size=VAL_SIZE / (1.0 - TEST_SIZE),
                                      random_state=42, stratify=y[idx_tv])

X_seq_train,  X_seq_val,  X_seq_test  = X_seq[idx_train],  X_seq[idx_val],  X_seq[idx_test]
X_flat_train, X_flat_val, X_flat_test = X_flat[idx_train], X_flat[idx_val], X_flat[idx_test]
y_train,      y_val,      y_test      = y[idx_train],      y[idx_val],      y[idx_test]

print(f'Train : {len(y_train):>8,}  pos={y_train.mean():.2%}')
print(f'Val   : {len(y_val):>8,}  pos={y_val.mean():.2%}')
print(f'Test  : {len(y_test):>8,}  pos={y_test.mean():.2%}')

In [ ]:
# ── Impute (mean) + Z-score normalise — fit on train only ────
feat_means = np.nanmean(X_flat_train, axis=0)
feat_stds  = np.nanstd(X_flat_train,  axis=0)
feat_stds[feat_stds == 0] = 1.0

def impute_scale(X, means, stds):
    X = X.copy()
    if X.ndim == 3:                      # (N, T, F)
        mask = np.isnan(X)
        X[mask] = np.take(means, np.where(mask)[2])
    else:                                # (N, F)
        for j in range(X.shape[1]):
            m = np.isnan(X[:, j])
            X[m, j] = means[j]
    return (X - means) / stds

X_seq_train_sc  = impute_scale(X_seq_train,  feat_means, feat_stds)
X_seq_val_sc    = impute_scale(X_seq_val,    feat_means, feat_stds)
X_seq_test_sc   = impute_scale(X_seq_test,   feat_means, feat_stds)
X_flat_train_sc = impute_scale(X_flat_train, feat_means, feat_stds)
X_flat_val_sc   = impute_scale(X_flat_val,   feat_means, feat_stds)
X_flat_test_sc  = impute_scale(X_flat_test,  feat_means, feat_stds)

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: cw[0], 1: cw[1]}
print(f'Class weights — 0: {cw[0]:.3f}  |  1: {cw[1]:.3f}')

## 6. Model 1 — Logistic Regression (Baseline)

In [ ]:
print('Training Logistic Regression...')
lr_model = LogisticRegression(
    C=1.0, max_iter=1000,
    class_weight='balanced',
    solver='lbfgs', random_state=42
)
lr_model.fit(X_flat_train_sc, y_train)

lr_prob_test = lr_model.predict_proba(X_flat_test_sc)[:, 1]
lr_pred_test = (lr_prob_test >= 0.5).astype(int)

lr_auroc = roc_auc_score(y_test, lr_prob_test)
lr_auprc = average_precision_score(y_test, lr_prob_test)
lr_f1    = f1_score(y_test, lr_pred_test, zero_division=0)

print(f'\nLogistic Regression — Test Set')
print(f'  AUROC : {lr_auroc:.4f}')
print(f'  AUPRC : {lr_auprc:.4f}')
print(f'  F1    : {lr_f1:.4f}')
print(classification_report(y_test, lr_pred_test,
                            target_names=['Non-Sepsis', 'Sepsis'],
                            zero_division=0))

In [ ]:
# Feature importance
coef = pd.Series(lr_model.coef_[0], index=FINAL_FEATURES)
top15 = coef.abs().sort_values(ascending=False).head(15)
colors_bar = ['#D85A30' if coef[f] > 0 else '#378ADD' for f in top15.index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top15.index[::-1], top15.values[::-1], color=colors_bar[::-1], alpha=0.85)
ax.set_xlabel('|Coefficient|')
ax.set_title('Logistic Regression — Top 15 Feature Importances')
ax.legend(handles=[
    mpatches.Patch(color='#D85A30', alpha=0.85, label='Risk-increasing'),
    mpatches.Patch(color='#378ADD', alpha=0.85, label='Protective')
])
plt.tight_layout()
plt.savefig('lr_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model 2 — LSTM (PyTorch)

**Why PyTorch instead of NumPy?**
- Real backpropagation through time (BPTT)
- Fully vectorised batched forward pass — no Python loops per sample
- Automatic GPU / Apple MPS acceleration
- Trains in **minutes** instead of hours

In [ ]:
class LSTMModel(nn.Module):
    """
    Stacked LSTM with dropout and a linear classification head.
    Input : (batch, seq_len, input_size)
    Output: (batch,) — raw logits (use BCEWithLogitsLoss during training)
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)          # (batch, seq_len, hidden)
        last   = out[:, -1, :]         # take last timestep
        return self.fc(self.dropout(last)).squeeze(1)  # (batch,)


def train_lstm(model, X_train, y_train, X_val, y_val,
               epochs=30, batch_size=256, lr=1e-3,
               pos_weight_val=None, device=DEVICE, verbose=True):
    """
    Full training loop with:
    - BCEWithLogitsLoss + class-imbalance pos_weight
    - Adam optimiser
    - ReduceLROnPlateau scheduler
    - Val AUROC tracking every epoch
    """
    # Tensors
    Xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=batch_size, shuffle=True,
                        num_workers=0, pin_memory=(device.type == 'cuda'))

    Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val, dtype=torch.float32).to(device)

    pw = torch.tensor([pos_weight_val], dtype=torch.float32).to(device) \
         if pos_weight_val else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=False
    )

    model.to(device)
    train_losses, val_aurocs = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(Xb)
            loss   = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(yb)

        avg_loss = epoch_loss / len(y_train)
        train_losses.append(avg_loss)

        # Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(Xv)
            val_probs  = torch.sigmoid(val_logits).cpu().numpy()
        auc = roc_auc_score(y_val, val_probs) if len(np.unique(y_val)) > 1 else float('nan')
        val_aurocs.append(auc)
        scheduler.step(auc)

        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(f'Epoch {epoch+1:3d}/{epochs}  '
                  f'loss={avg_loss:.4f}  val_AUROC={auc:.4f}  '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e}')

    return train_losses, val_aurocs


print('LSTM model and training loop ready.')
print(f'Architecture: Input({len(FINAL_FEATURES)}) → '
      f'LSTM({HIDDEN_SIZE} x {NUM_LAYERS} layers) → Dropout({DROPOUT}) → Linear(1)')

In [ ]:
print('Training PyTorch LSTM...')
print(f'  Device: {DEVICE}')
print(f'  seq_len={SEQUENCE_LEN}h | epochs={EPOCHS} | batch={BATCH_SIZE} | lr={LEARNING_RATE}')
print('-' * 60)

lstm_model = LSTMModel(
    input_size=len(FINAL_FEATURES),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

# pos_weight = ratio of negatives to positives (handles class imbalance)
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f'  pos_weight: {pos_weight:.2f}')
print('-' * 60)

train_losses, val_aurocs = train_lstm(
    lstm_model,
    X_seq_train_sc, y_train,
    X_seq_val_sc,   y_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    pos_weight_val=pos_weight,
    device=DEVICE,
    verbose=True
)
print('\nTraining complete.')

In [ ]:
# ── Test set evaluation ───────────────────────────────────────
lstm_model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_seq_test_sc, dtype=torch.float32).to(DEVICE)
    lstm_prob_test = torch.sigmoid(lstm_model(X_test_t)).cpu().numpy()

lstm_pred_test = (lstm_prob_test >= 0.5).astype(int)

lstm_auroc = roc_auc_score(y_test, lstm_prob_test)
lstm_auprc = average_precision_score(y_test, lstm_prob_test)
lstm_f1    = f1_score(y_test, lstm_pred_test, zero_division=0)

print(f'PyTorch LSTM — Test Set')
print(f'  AUROC : {lstm_auroc:.4f}')
print(f'  AUPRC : {lstm_auprc:.4f}')
print(f'  F1    : {lstm_f1:.4f}')
print(classification_report(y_test, lstm_pred_test,
                            target_names=['Non-Sepsis', 'Sepsis'],
                            zero_division=0))

## 8. Comparison & Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Model Comparison: PyTorch LSTM vs Logistic Regression',
             fontsize=14, fontweight='bold')

# ROC
ax = axes[0, 0]
for prob, lbl, col in [
    (lr_prob_test,   f'LR   (AUROC={lr_auroc:.3f})',   '#378ADD'),
    (lstm_prob_test, f'LSTM (AUROC={lstm_auroc:.3f})', '#D85A30')
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=col, lw=2, label=lbl)
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4,label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves'); ax.legend()

# Precision-Recall
ax = axes[0, 1]
for prob, lbl, col in [
    (lr_prob_test,   f'LR   (AUPRC={lr_auprc:.3f})',   '#378ADD'),
    (lstm_prob_test, f'LSTM (AUPRC={lstm_auprc:.3f})', '#D85A30')
]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ax.plot(rec, prec, color=col, lw=2, label=lbl)
ax.axhline(y_test.mean(), color='k', linestyle='--', lw=1, alpha=0.4,
           label=f'Baseline ({y_test.mean():.2%})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves'); ax.legend()

# Metric bar
ax = axes[0, 2]
mnames  = ['AUROC', 'AUPRC', 'F1']
lr_v    = [lr_auroc,   lr_auprc,   lr_f1]
lstm_v  = [lstm_auroc, lstm_auprc, lstm_f1]
x, w    = np.arange(3), 0.3
ax.bar(x-w/2, lr_v,   w, color='#378ADD', alpha=0.85, label='Logistic Regression')
ax.bar(x+w/2, lstm_v, w, color='#D85A30', alpha=0.85, label='LSTM')
for i,(lv,sv) in enumerate(zip(lr_v, lstm_v)):
    ax.text(i-w/2, lv+0.01, f'{lv:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i+w/2, sv+0.01, f'{sv:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(mnames)
ax.set_ylim(0, 1.15); ax.set_title('Metrics'); ax.legend()

# Confusion matrices
for ax, pred, title, cmap in [
    (axes[1,0], lr_pred_test,   'Confusion Matrix — LR',   'Blues'),
    (axes[1,1], lstm_pred_test, 'Confusion Matrix — LSTM', 'Oranges')
]:
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, pred),
        display_labels=['Non-Sepsis','Sepsis']
    ).plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(title)

# Training curves
ax = axes[1, 2]
epochs_x = range(1, len(train_losses) + 1)
ax.plot(epochs_x, train_losses, color='#D85A30', lw=2, label='Train Loss')
ax2 = ax.twinx()
ax2.plot(epochs_x, val_aurocs, 'o--', color='#1D9E75', lw=2, label='Val AUROC')
ax2.set_ylabel('Val AUROC', color='#1D9E75')
ax2.legend(loc='lower right')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('LSTM Training Curves'); ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_comparison.png')

In [ ]:
# Score distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Predicted Probability Distributions', fontsize=13, fontweight='bold')
for ax, probs, title in [
    (axes[0], lr_prob_test,   'Logistic Regression'),
    (axes[1], lstm_prob_test, 'PyTorch LSTM')
]:
    ax.hist(probs[y_test==0], bins=50, alpha=0.6, color='#378ADD', density=True, label='Non-Sepsis')
    ax.hist(probs[y_test==1], bins=50, alpha=0.6, color='#D85A30', density=True, label='Sepsis')
    ax.axvline(0.5, color='k', linestyle='--', lw=1.5, label='Threshold=0.5')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Density')
    ax.set_title(title); ax.legend()
plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Results Summary

In [ ]:
results = pd.DataFrame({
    'Model':     ['Logistic Regression', 'PyTorch LSTM'],
    'AUROC':     [lr_auroc,   lstm_auroc],
    'AUPRC':     [lr_auprc,   lstm_auprc],
    'F1':        [lr_f1,      lstm_f1],
    'Recall':    [recall_score(y_test, lr_pred_test,   zero_division=0),
                  recall_score(y_test, lstm_pred_test, zero_division=0)],
    'Precision': [precision_score(y_test, lr_pred_test,   zero_division=0),
                  precision_score(y_test, lstm_pred_test, zero_division=0)],
}).set_index('Model').round(4)

print('=' * 58)
print('FINAL RESULTS SUMMARY')
print('=' * 58)
print(results.to_string())
print('=' * 58)
winner = 'PyTorch LSTM' if lstm_auroc > lr_auroc else 'Logistic Regression'
print(f'\nBest model (AUROC): {winner}  (Δ={abs(lstm_auroc-lr_auroc):.4f})')
print(f'Prediction horizon : {PREDICTION_HORIZON}h | Lookback: {SEQUENCE_LEN}h')
print(f'Device used        : {DEVICE}')

In [ ]:
# ── Save model checkpoint ─────────────────────────────────────
torch.save({
    'model_state_dict': lstm_model.state_dict(),
    'feat_means': feat_means,
    'feat_stds':  feat_stds,
    'features':   FINAL_FEATURES,
    'config': {
        'input_size':  len(FINAL_FEATURES),
        'hidden_size': HIDDEN_SIZE,
        'num_layers':  NUM_LAYERS,
        'dropout':     DROPOUT,
        'seq_len':     SEQUENCE_LEN,
        'horizon':     PREDICTION_HORIZON,
    }
}, 'lstm_sepsis_checkpoint.pt')
print('Model checkpoint saved to lstm_sepsis_checkpoint.pt')
print('\nTo reload:')
print('  ckpt  = torch.load("lstm_sepsis_checkpoint.pt")')
print('  model = LSTMModel(**{k: ckpt["config"][k] for k in ["input_size","hidden_size","num_layers","dropout"]})')
print('  model.load_state_dict(ckpt["model_state_dict"])')